# AI on-call triage assistant — multi-agent LangChain + LangGraph + Claude Opus 4.8 on Oracle AI Database

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/langchain_ecosystem/ai_oncall_triage_langchain.ipynb)

-----

## The use case

Every AI platform team has the same problem: an issue lands in the queue ("agent is looping", "RAG isn't finding the docs", "Anthropic 429s under load"), and the on-call engineer has to decide — *fast* — three things at once:

1. **Has this happened before?** The closest past issues are the single most useful signal. Cosine search over an issue corpus surfaces them whether or not the new issue uses the same words — and whether each one is still open or already closed tells you if there's precedent for a fix.
2. **What does the standing triage policy say?** Severity thresholds (P0/P1/P2/P3), label-to-team routing, escalation paths. The rules don't change per incident; they shouldn't be re-derived per incident.
3. **What are the on-call engineer's preferences?** Some engineers prefer to take agent-loop issues themselves; some always route retrieval to the search-infra team. Saved memories outlive any single ticket.

A LangGraph **supervisor** orchestrates two specialists — `issue_analyst` (vector search over the issue corpus) and `policy_agent` (triage policy + on-caller prefs) — and synthesises a recommendation: **severity, label, suggested first-responder, similar past issues to reference, and a one-line first action.**

Agent state lives in a single Oracle AI Database: vector knowledge via `OracleVS`, long-term cross-thread memory via `AsyncOracleStore`, and per-thread checkpoints via `AsyncOracleSaver`. The same database also hosts an application-side LLM optimization (`OracleSemanticCache`) and standalone chat history (`OracleChatMessageHistory`). Live chat runs on **Claude Opus 4.8** with adaptive thinking.

### The triage workflow at a glance

```mermaid
flowchart TD
    issue["🐛 New issue lands in the queue<br/>(title + body + on-call user_id)"] --> sup["🧭 Triage supervisor<br/>(Claude Opus 4.8)"]
    sup -->|"1 · has this happened before?"| ia["issue_analyst<br/>semantic search over past issues"]
    sup -->|"2 · what does policy say?<br/>3 · what are the on-caller's prefs?"| pa["policy_agent<br/>triage policy + saved preferences"]
    ia -->|"similar issues + open/closed outcomes"| sup
    pa -->|"policy thresholds + engineer prefs"| sup
    sup --> rec["📋 Triage recommendation<br/>severity · label · first-responder ·<br/>similar issues · first action"]
```

The three questions an on-call engineer juggles map onto two specialists; the supervisor asks each, then synthesises one answer. Every arrow that touches data is backed by Oracle — the sections below build each piece bottom-up.

### Why this agent needs a semantic cache *and* durable chat history

Two of those Oracle primitives aren't about *what the agent knows* — they're about making the agent **cheap to run** and **coherent over time**. They're the easiest pieces to skip while prototyping and the most painful to retrofit in production, so it's worth being explicit about why a developer building this triage assistant needs each one:

- **Semantic cache (`OracleSemanticCache`) — because on-call work is repetitive and LLM calls are the bill.** The same class of incident lands again and again — "parquet load fails", "the agent is looping", "429s under load" — each filed in slightly different words. And a single triage run isn't one LLM call: it's *several* Claude Opus 4.8 calls (the supervisor, each specialist, and the final synthesis), so re-deriving a recommendation for a reworded duplicate is the single most expensive thing this system does. A semantic cache matches on **meaning**, not exact text, so a paraphrased recurrence of an already-solved incident comes back from one vector lookup in milliseconds instead of a fresh multi-agent reasoning loop that costs seconds and real tokens. In a queue where the same failures recur weekly, that is where nearly all of your latency and cost savings live — and it's why we wire the cache in even though the demo agents themselves run uncached.

- **Durable chat history (`OracleChatMessageHistory`) — because triage is a conversation, not a one-shot.** An engineer rarely stops at the first recommendation: *"what about the macOS case?"*, *"who did we route the last parquet issue to?"*, *"escalate it to P1."* The agent can only answer those coherently if it can see the earlier turns. Persisting that transcript in Oracle — rather than in process memory — means the conversation survives a kernel restart, a page refresh, or a **shift handoff**: the next on-call engineer opens the same `session_id` and picks up exactly where the last one left off. That is the difference between a stateless endpoint and an assistant a team can actually lean on during an incident.

Because both live in the **same** database as the vector store and the agent's own memory, they're one more table to provision, secure, and back up — not two more services to run.

## The stack: PALO + a pluggable vectorizer

This notebook is built on the **PALO stack** — the *Palo Alto Agentic Stack*: **P**ython · **A**nthropic (Claude) · **L**angChain / LangGraph · **O**racle AI Database. A multi-agent system needs one more interchangeable part — an **embedding model** — sourced here from OpenAI:

| Layer | Here | Role |
|-------|------|------|
| **P** — Python | the notebook runtime | glue |
| **A** — Anthropic | `ChatAnthropic` → **Claude Opus 4.8** | the reasoning engine: the supervisor and both specialists |
| **L** — LangChain / LangGraph | `langchain`, `langgraph`, `langgraph-supervisor`, `langchain-oracledb`, `langgraph-oracledb` | the agents, the supervisor graph, and every Oracle-backed persistence primitive |
| **O** — Oracle | Oracle AI Database 23ai/26ai | one substrate for vectors, long-term memory, checkpoints, the LLM cache, and chat history |
| *vectorizer* | **OpenAI** `text-embedding-3-small` | turns issue text into 1536-dim vectors (Anthropic ships no embedding endpoint) |

The whole point of this notebook is the **O** column: five kinds of agent state that teams usually scatter across five systems, all living in one database.

## What you'll learn

By the end of this notebook you'll have built a working multi-agent system and seen how each piece of agent state maps onto a single Oracle AI Database. Concretely:

- **Multi-agent orchestration** — the LangGraph *supervisor* pattern: one orchestrator LLM that delegates to focused specialist agents and synthesises their answers, instead of one monolithic prompt.
- **Agent persistence and caching**, and which Oracle primitive backs each: a **vector knowledge base** (`OracleVS`), **long-term cross-thread memory** (`AsyncOracleStore`), **short-term per-thread checkpoints** (`AsyncOracleSaver`), an application-side **semantic LLM cache** (`OracleSemanticCache`), and standalone **chat history** (`OracleChatMessageHistory`).
- **Why vector search beats keyword search for triage** — a hands-on demo where a substring filter misses what cosine similarity finds.
- **How to tune the chat layer for Claude Opus 4.8** — adaptive thinking, prompt caching, and the reasoning/embedding provider split.

The pattern transfers directly to any "retrieve precedent → apply policy → respect operator preferences → recommend" workflow: incident response, support-ticket routing, code-review triage, claims adjudication.

## Prerequisites

| You need | Notes |
|----------|-------|
| **Python 3.10+** | The `langchain` / `langgraph` 1.x APIs used here require it. |
| **A running Oracle AI Database** | The pinned `gvenzl/oracle-free:23.26.2` image below exposes `FREEPDB1` on `localhost:1521`, which is the default this notebook assumes. |
| **An Anthropic API key** | For Claude Opus 4.8 (the reasoning model). Get one at [console.anthropic.com](https://console.anthropic.com/). |
| **An OpenAI API key** | For `text-embedding-3-small` embeddings (Anthropic has no embedding endpoint). |

> **Start Oracle locally.** This command pins the tested image, creates the notebook user during first startup, and gives Docker an explicit database health check:
>
> ```bash
> docker run -d --name oracle-free \
>   -p 1521:1521 \
>   -e ORACLE_PASSWORD=OraclePwd_2026 \
>   -e APP_USER=NOTEBOOK_USER \
>   -e APP_USER_PASSWORD=NotebookPwd_2026 \
>   --health-cmd='healthcheck.sh' \
>   --health-interval=10s --health-timeout=5s \
>   --health-start-period=5m --health-retries=10 \
>   gvenzl/oracle-free:23.26.2
> ```
>
> Wait until `docker inspect --format '{{.State.Health.Status}}' oracle-free` prints `healthy`. The image provisions `NOTEBOOK_USER` in `FREEPDB1`; no manual SQL setup is needed. The first startup can take several minutes.
>
> This is designed for a local Jupyter kernel. A hosted Colab kernel cannot reach a database at your laptop's `localhost`; use an externally reachable Oracle service there or run the notebook locally.

# Part 1 — Environment setup

Two small but necessary steps before we touch the database: install the
right Python packages, then load the credentials the rest of the notebook
will use. After this part the kernel knows where all three services live —
**Oracle** (data + agent state), **Anthropic** (the chat model), and
**OpenAI** (embeddings).

## 1. Install packages

One `pip install`, latest releases. Each package earns its place:

| Package | What it is | Role in this notebook |
|---------|-----------|-----------------------|
| [`langchain`](https://pypi.org/project/langchain/) | Core LangChain | `create_agent` — builds each ReAct specialist. |
| [`langgraph`](https://pypi.org/project/langgraph/) | Agent runtime | the state-machine engine under the agents and the supervisor. |
| [`langgraph-supervisor`](https://pypi.org/project/langgraph-supervisor/) | Supervisor helper | `create_supervisor` — the multi-agent orchestration pattern. |
| [`langgraph-oracledb`](https://pypi.org/project/langgraph-oracledb/) | Oracle-backed LangGraph persistence | `AsyncOracleStore` (long-term memory) + `AsyncOracleSaver` (checkpoints). |
| [`langchain-oracledb`](https://pypi.org/project/langchain-oracledb/) | Oracle LangChain integration | `OracleVS`, `OracleSemanticCache`, `OracleChatMessageHistory`. |
| [`langchain-anthropic`](https://docs.langchain.com/oss/python/integrations/chat/anthropic) | Anthropic models for LangChain | `ChatAnthropic` → Claude Opus 4.8, the reasoning model. |
| [`langchain-openai`](https://docs.langchain.com/oss/python/integrations/text_embedding/openai) | OpenAI models for LangChain | `OpenAIEmbeddings` (`text-embedding-3-small`) for the vectors. |
| [`datasets`](https://pypi.org/project/datasets/) | Hugging Face data loader | pulls the real GitHub-issues corpus the agent searches. |
| [`oracledb`](https://pypi.org/project/oracledb/) | Python driver for Oracle | opens the sync + async connections everything shares. |


In [1]:
%pip install -qU \
    langchain langgraph langgraph-supervisor langgraph-oracledb \
    langchain-oracledb langchain-anthropic langchain-openai \
    oracledb datasets
print("✅ Latest releases installed. Restart the kernel if pip replaced an already-imported package.")

Note: you may need to restart the kernel to use updated packages.
✅ Latest releases installed. Restart the kernel if pip replaced an already-imported package.


Inject your API keys and Oracle credentials so the rest of the notebook
can read them from `os.environ`. In a real deployment you would source
these from a secrets manager rather than the notebook itself — the inline
form here is just for workshop ergonomics.

In [ ]:
import os

# Anthropic key for Claude Opus 4.8 (chat LLM) — paste yours below or
# set it in your shell. Get one at https://console.anthropic.com/.
# os.environ["ANTHROPIC_API_KEY"] = "sk-"

# OpenAI key — still required because we use OpenAI's text-embedding-3-small
# for OracleVS retrieval. Anthropic does not provide embedding endpoints.
# os.environ["OPENAI_API_KEY"] = "sk-pr"

# Workshop defaults match the APP_USER created by the Docker command above.
# Values already exported in your shell take precedence.
os.environ.setdefault("ORACLE_USER", "NOTEBOOK_USER")
os.environ.setdefault("ORACLE_PASSWORD", "NotebookPwd_2026")
os.environ.setdefault("ORACLE_DSN", "localhost:1521/FREEPDB1");


## 2. Environment variables and credentials

The previous cell injected your API keys and Oracle credentials into
`os.environ`. This cell reads them, validates them, and binds the Python
variables the rest of the notebook uses.

### Two providers, two jobs

This notebook deliberately splits the work across two model providers, and
it's worth understanding *why* — it's a pattern you'll repeat in production:

| Role | Provider | Model | Why |
|------|----------|-------|-----|
| **Reasoning / chat** | Anthropic | `claude-opus-4-8` | The supervisor and specialists do multi-step planning and synthesis. Opus 4.8's adaptive thinking + better tool triggering are exactly what a multi-agent loop needs. |
| **Embeddings** | OpenAI | `text-embedding-3-small` (1536-dim) | Anthropic does not ship an embedding endpoint. Retrieval quality depends on the embedding model, so we fix it here and never change it — every vector in Oracle lives in the same space. |

If you'd rather keep everything in-database, swap `OpenAIEmbeddings` for
`OracleEmbeddings` later — the only constraint is that the *same* embedding
model is used for both ingestion and query (otherwise cosine distance is
meaningless).

### Oracle credentials

`ORACLE_USER` + `ORACLE_PASSWORD` + `ORACLE_DSN` (split form, recommended),
or the combined form `user/password@host:port/service_name`.

### Why Claude Opus 4.8 for this notebook

Opus 4.8 (released 2026-05-28) is a drop-in upgrade for the supervisor pattern below. The features we explicitly lean on:

| Feature | What we use it for |
|---|---|
| **Adaptive thinking** | The supervisor's dispatch turns are shallow — it routes to `policy_agent` or `issue_analyst` and yields. The synthesis turn is heavy — it has to reconcile the triage policy, the on-caller's saved preferences, and the similar past issues into one recommendation. With `thinking={\"type\": \"adaptive\"}` the model thinks only when it judges the turn needs it; we set this default in §1's helper. |
| **Prompt caching (4096-token minimum cacheable prefix)** | Every supervisor turn re-sends the same supervisor prompt + 2 specialist system prompts + their tool specs. That payload comfortably clears Opus 4.8's 4096-token minimum cacheable prefix, so adding a `cache_control` breakpoint to the stable prefix yields up to 90% off cached input tokens on repeat turns. Note caching is opt-in — neither the API nor `create_supervisor` adds the annotation for you. |
| **Better tool triggering** | Fewer cases of skipped tool calls on tasks that require them. `issue_analyst` reliably invokes `search_similar_issues` instead of guessing; `policy_agent` reliably invokes both `get_triage_policy` and `get_user_memory`. |
| **1M-token context window** | Headroom for long checkpointed threads. `AsyncOracleSaver` can keep many turns in memory without forcing compaction, and the supervisor's view of history stays intact across a multi-day incident. |
| **`effort: high` default + 128k max output** | The synthesis step can produce a detailed recommendation without bumping into a small output cap. We cap at 8192 in `chat_model_kwargs` to keep the demo snappy; raise as needed. |

Source: [What's new in Claude Opus 4.8](https://platform.claude.com/docs/en/about-claude/models/whats-new-claude-4-8).

The cell below wires up the **reasoning** provider. It reads your `ANTHROPIC_API_KEY` from the environment — prompting once with `getpass` if it's missing, so the key is never hard-coded into the notebook — and pins `LLM_MODEL` to `claude-opus-4-8`. Because it uses `os.environ.setdefault`, a value already exported in your shell wins, which lets you point the whole notebook at a different Claude tier without editing code.

In [3]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

LLM_PROVIDER = "anthropic"
LLM_MODEL = os.environ.setdefault("LLM_MODEL", "claude-opus-4-8")
print(f"Chat model: {LLM_PROVIDER}/{LLM_MODEL}")

Chat model: anthropic/claude-opus-4-8


The helper below centralises the `ChatAnthropic` settings so every agent is built the same way. Two choices matter:

- **`max_tokens=8192`** — a generous output budget for the final synthesis step (Opus 4.8 supports up to 128k; we don't need that here).
- **`thinking={"type": "adaptive"}`** — lets Opus 4.8 decide *per turn* whether to reason. Shallow turns (the supervisor dispatching to a specialist) answer directly; the heavy synthesis turn thinks first. That spends thinking tokens only where they change the answer. Defining it once here means every specialist and the supervisor inherit the same tuned model.

In [4]:
def chat_model_kwargs(**extra):
    """Build kwargs for `ChatAnthropic` tuned for Claude Opus 4.8.

    Defaults pick up Opus 4.8's new capabilities:

    - `max_tokens=8192` — generous output budget for multi-agent
      synthesis. Opus 4.8 supports up to 128k output tokens; we don't
      need that here but the headroom is there if a recommendation
      grows long.
    - `thinking={"type": "adaptive"}` — the model decides per-turn
      whether to reason. On shallow turns (the supervisor dispatching
      to a specialist, or `search_similar_issues` returning a row)
      it responds directly. On the final synthesis step it reasons
      before answering. The result is fewer wasted thinking tokens
      vs. always-on extended thinking, at the same final quality.

    Prompt caching on Opus 4.8 requires an explicit `cache_control`
    breakpoint and a cacheable prefix of at least 4096 tokens — it is
    not enabled automatically. If repeated supervisor turns become a
    cost concern, add a cache_control breakpoint to the largest stable
    prefix (system prompt + tool specs).
    """
    kwargs = dict(
        max_tokens=8192,
        thinking={"type": "adaptive"},
    )
    kwargs.update(extra)
    return kwargs


The chat model is Claude Opus 4.8, but embeddings stay on OpenAI's
`text-embedding-3-small` (1536-dim). 

Pinning the embedding model is a
correctness requirement, not a preference: a vector store is only coherent
if the vectors written at ingestion time and the vectors computed at query
time come from the *same* model. 

Mixing embedding models silently degrades
every similarity search in the notebook.

In [5]:
# ─── Embeddings — always OpenAI ────────────────────────────────────────────
# Anthropic ships no embedding endpoint, and retrieval quality depends on a
# single, fixed embedding model — so every vector in Oracle lives in one space.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMS = 1536
print(f"Embedding model: openai/{EMBEDDING_MODEL} ({EMBEDDING_DIMS} dims)")

Embedding model: openai/text-embedding-3-small (1536 dims)


Last piece of setup: read and validate the Oracle connection details.

We accept either the recommended *split* form (`ORACLE_USER` +
`ORACLE_PASSWORD` + `ORACLE_DSN` tail) or the *combined* form
(`user/pw@host:port/service_name`), then print a one-line summary of every
configuration value the rest of the notebook will rely on.

In [6]:
# ─── Oracle credentials ────────────────────────────────────────────────────
ORACLE_USER = os.environ.get("ORACLE_USER")
ORACLE_PASSWORD = os.environ.get("ORACLE_PASSWORD")
ORACLE_DSN = os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1")

if "@" in ORACLE_DSN and not (ORACLE_USER and ORACLE_PASSWORD):
    user_pw, ORACLE_DSN = ORACLE_DSN.split("@", 1)
    if "/" not in user_pw:
        raise ValueError(
            "ORACLE_DSN contains `@` but no user/password before it. Expected "
            "`user/password@host:port/service_name`, or set ORACLE_USER + "
            "ORACLE_PASSWORD + ORACLE_DSN separately."
        )
    ORACLE_USER, ORACLE_PASSWORD = user_pw.split("/", 1)

if not (ORACLE_USER and ORACLE_PASSWORD):
    raise ValueError(
        "Oracle credentials missing. Set ORACLE_USER + ORACLE_PASSWORD, or set "
        "ORACLE_DSN to the combined form `user/password@host:port/service_name`."
    )

print(f"Chat LLM:   {LLM_PROVIDER}/{LLM_MODEL} (adaptive thinking, max_tokens=8192)")
print(f"Embeddings: openai/{EMBEDDING_MODEL} ({EMBEDDING_DIMS} dims)")
print(f"Oracle:     {ORACLE_USER}@{ORACLE_DSN}")

Chat LLM:   anthropic/claude-opus-4-8 (adaptive thinking, max_tokens=8192)
Embeddings: openai/text-embedding-3-small (1536 dims)
Oracle:     NOTEBOOK_USER@localhost:1521/FREEPDB1


> **Key takeaway — Part 1.** The notebook talks to three services: **Oracle**
> (data + agent state), **Anthropic** (Claude Opus 4.8, the reasoning model),
> and **OpenAI** (`text-embedding-3-small`, embeddings only). Reasoning and
> embeddings live with different providers on purpose — Anthropic ships no
> embedding endpoint, and retrieval quality depends on a fixed embedding
> model. From here on the notebook uses the Python variables we just bound
> (`ORACLE_USER`, `LLM_MODEL`, `EMBEDDING_MODEL`, …) rather than re-reading
> `os.environ`.

# Part 2 — Oracle AI Database as the agent's persistence substrate

The agent we're about to build has **three kinds of persisted state**, plus
a model-call cache; every one lives in Oracle:

| State | Primitive | Package |
|-------|-----------|---------|
| Vector knowledge base (issue reports + policy) | `OracleVS` | `langchain-oracledb` |
| Long-term cross-thread memory (user preferences) | `AsyncOracleStore` | `langgraph-oracledb` |
| Per-thread short-term memory (checkpoints) | `AsyncOracleSaver` | `langgraph-oracledb` |
| LLM call cache | `OracleSemanticCache` | `langchain-oracledb` (Part 3) |

This part wires up the first three. The cache joins them in Part 3 on the
same Oracle client.

## 3. Open the Oracle connections

`oracledb` exposes both **sync** and **async** connection objects, and the
two families cannot share an underlying socket — async API calls require an
`AsyncConnection`, sync API calls require a `Connection`. So we open one
of each:

- **`oracle_client`** (sync) — used by `OracleVS`, `OracleSemanticCache`,
  and `OracleChatMessageHistory`.
- **`saver_conn`** (async) — used by `AsyncOracleSaver`.
- **`store_conn`** (async) — used by `AsyncOracleStore`.

All three point at the same database, so from Oracle's perspective this is
one workload in one schema.

In [7]:
import oracledb
from langchain_openai import OpenAIEmbeddings

connect_kwargs = {"user": ORACLE_USER, "password": ORACLE_PASSWORD, "dsn": ORACLE_DSN}
# Connecting as SYS requires SYSDBA mode (ORA-28009).
if ORACLE_USER.lower() == "sys":
    connect_kwargs["mode"] = oracledb.AUTH_MODE_SYSDBA

oracle_client = oracledb.connect(**connect_kwargs)            # sync  — OracleVS, cache, chat history
saver_conn   = await oracledb.connect_async(**connect_kwargs) # async — AsyncOracleSaver
store_conn   = await oracledb.connect_async(**connect_kwargs) # async — AsyncOracleStore
print("opened 1 sync + 2 async Oracle connections")

opened 1 sync + 2 async Oracle connections


Now the embedding model. We build it **once** and reuse the same object for the vector store, the long-term store's index, and the semantic cache — so every vector written to Oracle lives in the same 1536-dim space (mixing embedding models silently breaks cosine similarity).

In [8]:
# One embedding model, reused everywhere vectors are written or queried.
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
print(f"embeddings ready: openai/{EMBEDDING_MODEL} ({EMBEDDING_DIMS} dims)")

embeddings ready: openai/text-embedding-3-small (1536 dims)


> **Key takeaway — Connections.** One database, three Python handles —
> because `oracledb` keeps sync and async cleanly separate. The same
> `embeddings` model is reused everywhere so every vector in Oracle lives
> in the same 1536-dim space.

## 4. Instantiate `OracleVS` — the vector knowledge base

`OracleVS` is the canonical LangChain vector store for Oracle. We give it:

- the **sync client** we just opened,
- the **embedding function** (every text we `add_texts` will be embedded with it),
- the **table name** (`oncall_issue_kb` — it holds both the historical issue
  reports *and* the triage policy memo),
- the **distance metric** (`COSINE`, the right choice for normalized
  text embeddings).

The backing table is created by the `OracleVS` constructor itself — it
makes a single embedding call to determine the vector dimension. At this
corpus size similarity search runs as an exact scan; no vector index is
needed (use `langchain_oracledb`'s `create_index` helper when the corpus
grows). We bind the table name to `VECTOR_TABLE` so later cells (the idempotent re-seed) can
reference the exact same table. We'll fill the store with issue reports and
a policy memo in Part 4.

In [9]:
from langchain_oracledb.vectorstores.utils import DistanceStrategy
from langchain_oracledb import OracleVS

# One table holds the whole knowledge base: past issue reports + the triage
# policy memo. Bind it to a name so the re-seed cell can target it exactly.
VECTOR_TABLE = "oncall_issue_kb"

oracle_vs = OracleVS(
    client=oracle_client,
    embedding_function=embeddings,
    table_name=VECTOR_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)
print(f"OracleVS ready (table: {VECTOR_TABLE}, distance: COSINE)")

OracleVS ready (table: oncall_issue_kb, distance: COSINE)


> **Key takeaway — `OracleVS`.** A standard LangChain `VectorStore`
> implementation, backed by Oracle Vector Search. It also acts as the
> **shared connection anchor** for everything sync in this notebook —
> the cache and chat history will reuse `oracle_vs.client`.

## 5. Instantiate `AsyncOracleStore` — long-term cross-thread memory

`AsyncOracleStore` implements LangGraph's `BaseStore` interface. 

Think of it as a hierarchical key-value store with an optional vector index. 

Agents read and write to it via the LangGraph `Store` API, and the data **persists
across `thread_id`s** — that's the "cross-thread" / "long-term" part.

We configure it with an HNSW index on the `"note"` field so the memories
themselves are semantically searchable, then call `.setup()` to create the
backing tables on first run.

> 📚 **Docs:** `AsyncOracleStore` implements LangGraph's long-term **Store** interface — the hierarchical, optionally vector-indexed key/value store agents use for cross-thread memory. See [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence) for the concept and the `put` / `search` API it exposes.

In [10]:
import uuid

from langgraph_oracledb.store.oracle import AsyncOracleStore

# A fixed suffix so the tables are stable across re-runs (teardown targets them).
STORE_TABLE_SUFFIX = "oncall_triage_memory"
agent_store = AsyncOracleStore(
    store_conn,
    index={
        "dims": EMBEDDING_DIMS,
        "embed": embeddings,
        "fields": ["note"],
        "index_type": {"type": "hnsw", "distance_metric": "COSINE"},
    },
    table_suffix=STORE_TABLE_SUFFIX,
)
await agent_store.setup()
print(f"AsyncOracleStore ready (long-term memory, table_suffix={STORE_TABLE_SUFFIX})")

AsyncOracleStore ready (long-term memory, table_suffix=oncall_triage_memory)


> **Key takeaway — `AsyncOracleStore`.** This is where facts the
> **agent should remember regardless of which conversation is happening**
> live. Examples: user preferences, organisation-wide policies, persona
> facts. Distinct from per-thread chat history (next cell).

## 6. Instantiate `AsyncOracleSaver` — per-thread checkpoints

`AsyncOracleSaver` is LangGraph's checkpointer. 

It snapshots **every step** of the agent graph (messages, intermediate tool outputs, the supervisor's
internal state) keyed by `thread_id`. 

Re-invoking the agent with the same
`thread_id` resumes from where it left off; a new `thread_id` starts
fresh. 

This is **short-term memory** — scoped to one conversation.

In [12]:
from langgraph_oracledb.checkpoint.oracle import AsyncOracleSaver

saver = AsyncOracleSaver(saver_conn)
await saver.setup()
print("AsyncOracleSaver ready (per-thread checkpoint state)")

AsyncOracleSaver ready (per-thread checkpoint state)


> **Key takeaway — `AsyncOracleSaver` vs `AsyncOracleStore`.**
>
> Both are persistence. 
>
> The **saver** holds *this conversation's* timeline
> (keyed by `thread_id`); the **store** holds *facts that outlive any single
> conversation* (keyed by your own namespace, e.g. `user_id`).

> **Key takeaway — Part 2.** 
>
> An agent has several *kinds* of state, and the
> usual mistake is to scatter them across a vector DB, a Redis, a Postgres,
> and a checkpoint file. 

> Here all three live in **one Oracle AI Database**:
> `OracleVS` (what the agent knows), `AsyncOracleStore` (what it remembers
> across conversations), and `AsyncOracleSaver` (where it is in *this*
> conversation). 
>
> One backup, one security boundary, one set of credentials —
> and, as Part 3 shows, the same database also does duty as the LLM cache.

# Part 3 — LLM call caching with `OracleSemanticCache`

LLM calls are slow and expensive. 

A semantic cache stores past
`(prompt, response)` pairs **by vector similarity of the prompt** — so even
paraphrased prompts can return the cached response. 

Oracle Vector Search
makes this a one-line wiring.

## 7. Configure the semantic cache

We hand it `oracle_vs.client` (so it reuses the same Oracle connection),
the `embeddings` model, a table name to hold cache entries, and a
`score_threshold` — vectors closer than this are considered "the same
prompt" for cache-hit purposes (smaller = stricter).

In [13]:
from langchain_oracledb.cache import OracleSemanticCache

semantic_cache = OracleSemanticCache(
    client=oracle_vs.client,
    embedding=embeddings,
    table_name="oncall_semantic_cache",
    distance_strategy=DistanceStrategy.COSINE,
    score_threshold=0.05,  # tighter = fewer near-miss false positives
)
print("OracleSemanticCache configured (table: oncall_semantic_cache)")

OracleSemanticCache configured (table: oncall_semantic_cache)


> **Key takeaway — caching scope.** We **configure** the cache but
> deliberately don't install it globally via `set_llm_cache`. The agents
> further down call their LLM uncached (we want to watch them reason every
> run); the next cell attaches the cache only to a dedicated demo chat model
> for a clean before/after timing test. Provider-side prompt caching and this
> application-side Oracle semantic cache are separate mechanisms.

## 8. Sanity check the cache — two timed calls

We clear the cache first so every run starts cold. 

Same prompt, two invocations. 

The first call goes to Anthropic; the second resolves from `OracleSemanticCache` without another model call. 

It still performs an embedding plus an Oracle lookup, so treat it as fast rather than literally instant.

In [14]:
import time

from langchain_anthropic import ChatAnthropic

semantic_cache.clear()  # start cold so the miss/hit timings are honest on re-runs

# The cache is attached to this one demo model only (not installed globally), so
# the agents further down still reason uncached on every run.
cache_demo_model = ChatAnthropic(model=LLM_MODEL, cache=semantic_cache, max_tokens=120)
PROMPT = "In one sentence: what is vector similarity search?"

**First call — the cache miss.** The model has never seen this prompt before, so the request reaches Claude. 

The `OracleSemanticCache` silently records the `(prompt, response)` pair on the way back, embedding the prompt for future similarity lookups.

In [15]:
t0 = time.perf_counter()
r1 = cache_demo_model.invoke(PROMPT)
miss = time.perf_counter() - t0


**Second call — same prompt.** The cache embeds the new prompt, finds
the exact prompt from the first call (distance zero, inside the threshold),
and returns the stored response *without* touching the LLM. This call
should be roughly the cost of one embedding round-trip plus a vector
search.

In [16]:
t0 = time.perf_counter()
r2 = cache_demo_model.invoke(PROMPT)
hit = time.perf_counter() - t0

Compare the two timings side by side and print the cached response we
just got back. The ratio is what the cache buys you — repeat (and even
paraphrased) prompts skip the LLM entirely.

In [17]:
print(f"first call  (miss): {miss:.2f}s")
print(f"second call ( hit): {hit:.2f}s")
print(f"speedup:            {miss / max(hit, 0.001):.1f}x")
print()
print(r2.content)
print("\nsecond identical prompt served from OracleSemanticCache:", r1.content == r2.content)

first call  (miss): 4.08s
second call ( hit): 0.22s
speedup:            18.9x

Vector similarity search is a technique for finding items whose numerical vector representations (embeddings) are closest to a query vector, typically measured by metrics like cosine similarity or Euclidean distance, enabling semantic matching based on meaning rather than exact keywords.

second identical prompt served from OracleSemanticCache: True


> **Key takeaway — Part 3: semantic cache wins.** Identical and near-duplicate
> prompts skip the model entirely. The observed speedup depends on model and
> embedding latency. Because matching is by embedding distance, sufficiently
> close **paraphrases may also hit** when they fall within `score_threshold`;
> we time the identical-prompt case here for a clean before/after. In a triage
> system where the same class of incident recurs weekly, that turns repeated
> reasoning into a vector lookup. The whole cache is just another table in the
> same Oracle database from Part 2.

# Part 4 — Ingest the issue corpus

The agents in Part 5 need three kinds of knowledge in Oracle before they can answer anything:

1. **Past issue reports** (title, body, labels, and open/closed state) in `OracleVS`, retrievable by semantic similarity to the new issue.
2. **The standing triage policy** in `OracleVS` — same store, retrieved the same way.
3. **On-call engineer preferences** in `AsyncOracleStore` — long-term memory keyed by `user_id`.

This part loads a small, realistic corpus of AI/agent issues, builds the docs, and seeds all three.

## 9. Load the GitHub issues dataset from Hugging Face

We use [`lewtun/github-issues`](https://huggingface.co/datasets/lewtun/github-issues) — ~3k real issues + PRs harvested from the public [`huggingface/datasets`](https://github.com/huggingface/datasets) repo. 

Each row has the shape a real GitHub issue export does: `id`, `number`, `title`, `body`, `labels`, `state` (`open` / `closed`), `is_pull_request`, `html_url`, timestamps, etc.

For the workshop we keep ~25 issues — filter out PRs, drop rows with empty bodies, and project the columns the triage agents actually need. 

In production you'd swap this for your own org's issues (via the GitHub API or a private mirror); the rest of the notebook doesn't care where the data came from.

> **A note on "resolutions."** This public export does **not** include the closing comments that actually resolved each issue — only the issue body and its `open`/`closed` **state**. 
> 
> That's still useful precedent (a *closed* near-duplicate means "this has happened and was fixed"), and it's what we index here. 
>
> In a production triage system you'd enrich each closed issue with its resolving comment / linked PR via the GitHub API, so the analyst can cite the *actual* fix, not just the fact that one exists. 
>
> We keep that out of scope to stay focused on the Oracle + multi-agent mechanics.

### What Part 4 builds

```mermaid
flowchart LR
    hf["🤗 lewtun/github-issues<br/>(~3k real issues)"] --> filt["filter: drop PRs<br/>+ empty bodies → keep 25"]
    filt --> rep["render each issue<br/>as one text report"]
    rep --> emb["embed with OpenAI<br/>text-embedding-3-small"]
    pol["standing triage policy memo"] --> emb
    emb --> vs[("OracleVS<br/>oncall_issue_kb")]
    prefs["engineer preferences<br/>alice / bob / carol"] --> store[("AsyncOracleStore")]
```

We pull a real corpus, turn each issue into one retrievable document, embed it into the **same vector space** as the triage policy, and store engineer preferences separately as long-term memory. The next cells do exactly this, top to bottom.

The cell below loads the corpus and **previews the first five issues as a DataFrame**, so you can see the exact fields each record carries — `id`, `title`, `state`, `labels` (plus the `body` and `url` we keep) — the raw material the agents will search.

In [18]:
from collections import Counter

from datasets import load_dataset

def _label_names(labels_field):
    """Each row's `labels` is a list of dicts; project just the name."""
    out = []
    for l in labels_field or []:
        if isinstance(l, dict) and l.get("name"):
            out.append(l["name"])
        elif isinstance(l, str):
            out.append(l)
    return out


def _is_pr(row):
    """A row is a pull request, not an issue. Prefer the explicit boolean
    column; fall back to the `pull_request` payload on older revisions."""
    if "is_pull_request" in row:
        return bool(row["is_pull_request"])
    return row.get("pull_request") is not None


# Real GitHub issues + PRs harvested from the public dataset.
raw = load_dataset("lewtun/github-issues", split="train")
print(f"loaded {len(raw)} rows from lewtun/github-issues")

RAW_ISSUES = [
    {
        "id": f"hf-datasets-{row['number']}",
        "title": row["title"],
        "body": (row["body"] or "")[:2000],
        "labels": _label_names(row["labels"]),
        "state": row["state"],
        "url": row["html_url"],
    }
    for row in raw
    if not _is_pr(row) and row["body"] and len(row["body"]) > 80
][:25]

label_counts = Counter(l for issue in RAW_ISSUES for l in issue["labels"])
state_counts = Counter(issue["state"] for issue in RAW_ISSUES)
print(f"kept {len(RAW_ISSUES)} issues  |  states: {dict(state_counts)}")
print(f"top labels: {label_counts.most_common(8)}")

import pandas as pd
print("\nPreview of the issues we'll index:")
display(pd.DataFrame(RAW_ISSUES)[["id", "title", "state", "labels"]].head())

/Users/richmondalake/Desktop/projects/OAMP/.venv313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.


loaded 3019 rows from lewtun/github-issues
kept 25 issues  |  states: {'closed': 15, 'open': 10}
top labels: [('bug', 17), ('dataset request', 4), ('enhancement', 2), ('streaming', 1)]


Pick the issues that go into the knowledge base. 

In production you'd rank by
comment count, reactions, time-to-resolution, or similarity to a currently
open issue. 

At this workshop scale every filtered issue is a real, distinct
incident worth indexing, so we use the full set.

In [20]:
# At this scale we keep every filtered issue. In production you'd rank by
# comment count, recency, label severity, or similarity to a currently-open issue.
top_issues = RAW_ISSUES
print(f"will index {len(top_issues)} issues into OracleVS")

will index 25 issues into OracleVS


> **Key takeaway — issue corpus is the agent's institutional memory.** Every past incident — its title, body, labels, and whether it's still open — becomes a retrievable doc. The fresher and richer the corpus (closing comments, linked PRs, time-to-resolve), the more useful the on-call experience.

## 10. Seed `OracleVS` with issue reports + triage policy

Two passes:

1. Each issue becomes one short markdown-ish report doc.
2. We append a single **triage policy memo** describing the standing rules — same store, same embedding space, retrieved the same way the agents will retrieve issue reports.

We truncate the table first so every re-run is deterministic.


In [22]:
import re

def report(issue):
    """Format one issue into a single retrievable doc."""
    labels = ", ".join(issue["labels"]) or "(none)"
    return (
        f"Issue {issue['id']} — {issue['title']}\n"
        f"State: {issue['state']} | Labels: {labels}\n"
        f"URL: {issue['url']}\n"
        f"Body: {issue['body']}"
    )

In [23]:
# Idempotent seed: clear any rows left from a prior run so every re-run rebuilds
# the knowledge base cleanly. The table was already created by the OracleVS
# constructor in Part 2; we tolerate ORA-00942 anyway in case it was dropped.
try:
    with oracle_client.cursor() as cur:
        # Oracle identifiers cannot be bind variables. Validate our constant,
        # then quote its normalized form so cleanup cannot become SQL injection.
        if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_$#]{0,127}", VECTOR_TABLE):
            raise ValueError(f"Unsafe Oracle table identifier: {VECTOR_TABLE!r}")
        # OracleVS creates this table as a quoted, case-sensitive name; keep
        # that exact case while quoting the already-validated identifier.
        quoted_vector_table = f'"{VECTOR_TABLE}"'
        cur.execute(f"DELETE FROM {quoted_vector_table}")
        oracle_client.commit()
    print(f"cleared existing rows from {VECTOR_TABLE}")
except oracledb.DatabaseError as exc:
    (error,) = exc.args
    if error.code == 942:  # ORA-00942: table or view does not exist
        print(f"{VECTOR_TABLE} does not exist yet — it will be created on first write")
    else:
        raise

cleared existing rows from oncall_issue_kb


Render each issue as a short report, append the standing triage policy memo, and push them all to `OracleVS` in one call.


### How the embeddings are generated — and what "semantic search" means here

Each issue is first flattened into a single text **report** (title + state + labels + body) by the `report()` helper above. That whole string is handed to OpenAI's `text-embedding-3-small`, which returns a **1536-dimensional vector** — a numeric fingerprint of the issue's *meaning* — and `OracleVS.add_texts` stores each vector next to its metadata in `oncall_issue_kb`.

Why embed the *report* (title + labels + body) rather than just the title? Because retrieval quality depends on signal: the body carries the symptom, the labels carry the category, the title carries the summary — embedding them together lets a query match on any of them.

**Semantic search** is then just: embed the incoming query with the *same* model, and ask Oracle for the stored vectors with the smallest cosine distance. A new report that says "crashes" still lands near an old one that said "raises an error," because the *vectors* are close even though the *words* aren't.

```mermaid
flowchart LR
    q["query:<br/>'parquet load crashes'"] --> qe["embed<br/>(text-embedding-3-small)"]
    doc["past issue:<br/>'load_dataset raises KeyError'"] --> de["embed<br/>(same model)"]
    qe --> cmp{{"cosine distance<br/>in Oracle"}}
    de --> cmp
    cmp --> near["nearest ⇒ same meaning,<br/>different words"]
```

In [24]:
texts = [report(issue) for issue in top_issues]
metadatas = [
    {"type": "issue_report", "issue_id": issue["id"],
     "state": issue["state"], "labels": issue["labels"], "url": issue["url"]}
    for issue in top_issues
]

texts.append(
    "On-call triage policy:\n"
    "- P0: data corruption or breaks all downstream users. Page primary + "
    "secondary on-call. Acknowledge within 15 min.\n"
    "- P1: major degradation or a popular dataset is unloadable. Assign to "
    "primary on-call. Resolve or escalate within 4 hours.\n"
    "- P2: bug with workaround. Triage by primary on-call; route to specialist "
    "by label.\n"
    "- P3: docs, enhancements, questions. Add to backlog; no SLA.\n"
    "- Label-to-team routing: 'bug' \u2192 primary on-call; 'documentation' \u2192 "
    "docs owner; 'enhancement' / 'good first issue' \u2192 community-PR sweep; "
    "'help wanted' \u2192 on-call lead for triage.\n"
    "- Always cite at least one prior similar issue (with its URL) before "
    "recommending a triage call."
)
metadatas.append({"type": "policy", "name": "triage_policy"})

oracle_vs.add_texts(texts, metadatas=metadatas)
print(f"OracleVS: {len(texts)} docs ({len(texts)-1} issues + 1 policy) ingested.")


OracleVS: 26 docs (25 issues + 1 policy) ingested.


> **Key takeaway — one store, multiple document types.** The triage policy and the issue reports live in the same `OracleVS` table. 
>
> Metadata distinguishes them, but the agents retrieve them through the same `similarity_search(...)` call. Adding a runbook later is one more `add_texts` row.


## 11. Seed `AsyncOracleStore` with on-call engineer preferences

Long-term memories live under hierarchical namespaces. 

We use `("users", <user_id>, "memories")` so each engineer's preferences are isolated. 

These are written **once**, retrieved by any future agent invocation on any `thread_id` — that's the cross-thread part.




### Writing long-term memories with `aput`

Each `agent_store.aput(...)` call below takes three positional arguments — we annotate them inline in the code so you can see the shape rather than infer it:

- **`namespace`** — a *tuple* that hierarchically scopes the memory, here `("users", <user_id>, "memories")`. Because Alice's rows live under `("users", "alice", "memories")`, a lookup for Bob never even scans them — isolation without row-level security.
- **`key`** — a stable string id for this specific memory (e.g. `"pref-bugs"`); writing the same key again overwrites it.
- **`value`** — a dict of the actual content. The store embeds the field named in the index config (`"note"`, set back in Part 2) so the memory is semantically searchable later by `get_user_memory`.

In [25]:
await agent_store.aput(
    ("users", "alice", "memories"),   # namespace: (root, user_id, kind) — hierarchical scope
    "pref-bugs",                       # key: stable id for this memory (writing it again overwrites)
    {"note": "Alice owns 'bug' label issues — anything where a dataset fails to "   # value: dict; the 'note' field is embedded
             "load, parse, or stream. Route bug-labelled issues to her by default."},
)
await agent_store.aput(
    ("users", "bob", "memories"),
    "pref-docs-enhancements",
    {"note": "Bob handles 'documentation' and 'enhancement' labels. Prefers to "
             "take docs requests and small DX improvements directly."},
)
await agent_store.aput(
    ("users", "carol", "memories"),
    "pref-on-call-lead",
    {"note": "Carol is on-call lead this rotation. Wants P0/P1 paged to her "
             "immediately; delegates P2/P3 to specialists by label."},
)
print("AsyncOracleStore: seeded engineer preferences for alice, bob, carol")

AsyncOracleStore: seeded engineer preferences for alice, bob, carol


> **Key takeaway — namespaces vs metadata.** 
> 
> The store keys memories by a tuple (`users` / `user_id` / `memories`), so retrieving Alice's preferences never even *scans* Bob's rows. 
>
> Each on-call engineer's signals stay disjoint without us writing row-level security.


## 12. The aha moment — naive substring vs semantic vector search

A new issue lands, described the way a frustrated user actually writes it:
*"loading a dataset from the hub **crashes** with a parquet error."*

The historical issue that explains this almost certainly exists in the
corpus — but it was filed using words like *"fails"*, *"raises"*, or
*"KeyError"*, not *"crashes"*. A substring/keyword filter searching for the
user's exact wording misses it. Semantic vector search matches on *meaning*,
so it surfaces the related parquet/loading issues regardless of vocabulary.

Watch the naive substring filter for the literal token `crashes` come up
short while OracleVS's semantic search returns the relevant past issues.

In [27]:
import re

# Realistic incoming-issue text — the user describes the symptom in their own
# words, not the canonical terms the original report happened to use.
QUERY_TEXT = "loading a dataset from the hub crashes with a parquet error"


### Why compare against a *naive substring* filter?

The next cell deliberately uses the crudest possible keyword approach — a Python substring/regex search for the user's exact word `crashes` — to make the failure of literal matching obvious. Real reports say "fails", "raises", or "error", so exact-token matching under-returns.

To be clear, this is a **strawman on purpose**. Oracle itself offers *real* keyword retrieval: `langchain-oracledb` can build an **Oracle Text** index and run `CONTAINS`-based full-text search, and even fuse it with vectors for **hybrid** search. A production system would benchmark vector search against *that*, not a substring filter. We use the substring here only to isolate the single idea we're teaching — matching on **meaning** vs matching on **surface tokens** — and the gap you'll see is the whole reason embeddings exist for triage.

In [28]:

# Naive: literal substring filter for the user's word "crashes" (skip the
# policy memo). Real issues say "fails"/"raises"/"error", so this under-returns.
naive = [m["issue_id"] for t, m in zip(texts[:-1], metadatas[:-1])
         if re.search(r"\bcrashes\b", t, re.IGNORECASE)]
print(f"NAIVE substring 'crashes' → {len(naive)} hits: {naive}")

# Semantic: search issue reports only; the policy row shares the table but
# OracleVS pushes this metadata filter into Oracle's JSON predicate.
hits = oracle_vs.similarity_search(QUERY_TEXT, k=5, filter={"type": "issue_report"})
print(f"\nSEMANTIC '{QUERY_TEXT}' → {len(hits)} hits:")
for d in hits:
    print(f"  {d.page_content.splitlines()[0]}")
print("\nEvery semantic hit is an issue report (policy row excluded by the metadata filter):",
      all(d.metadata.get("type") == "issue_report" for d in hits))

NAIVE substring 'crashes' → 1 hits: ['hf-datasets-2918']

SEMANTIC 'loading a dataset from the hub crashes with a parquet error' → 5 hits:
  Issue hf-datasets-2924 — "File name too long" error for file locks
  Issue hf-datasets-2941 — OSCAR unshuffled_original_ko: NonMatchingSplitsSizesError
  Issue hf-datasets-2923 — Loading an autonlp dataset raises in normal mode but not in streaming mode
  Issue hf-datasets-2921 — Using a list of multi-dim numpy arrays raises an error "can only convert 1-dimensional array values"
  Issue hf-datasets-2937 — load_dataset using default cache on Windows causes PermissionError: [WinError 5] Access is denied

Every semantic hit is an issue report (policy row excluded by the metadata filter): True


> **Key takeaway — embeddings beat keywords for triage.**
> 
> A new issue rarely uses the same vocabulary as the past issue that solved it. 
> 
> Vector search is the only thing that surfaces "this looks like RAG-208 from three months ago" when the new report doesn't mention RAG.


> **Key takeaway — Part 4.** The agent's knowledge is now in place: real
> GitHub issues + the triage policy share one semantically-searchable
> `OracleVS` table, and per-engineer preferences sit in `AsyncOracleStore`
> keyed by `user_id`. 
> 
> Two storage shapes, chosen deliberately — *unstructured,
> retrieve-by-meaning* knowledge goes in the vector store; *structured,
> retrieve-by-key* facts go in the namespaced store. Part 5 builds the agents
> that read from both.

# Part 5 — Building and running the multi-agent system

With the data in place we can now build the agents. This part is the
intellectual core of the notebook: we'll discuss **why** multi-agent,
**how** the supervisor pattern works, and **where** each kind of memory
plugs in. Then we'll build the tools, compile the agents, and run the
whole thing end-to-end.

## 13. Architecture

```mermaid
flowchart TD
    user["On-call request<br/>(new issue + user_id)"]
    user --> supervisor
    cache_demo["dedicated cache demo model<br/>(Part 3 only)"]

    subgraph multiAgent ["Multi-agent graph (langgraph-supervisor)"]
        direction TB
        supervisor{{"supervisor<br/>(orchestrator LLM)"}}
        issue_analyst["issue_analyst<br/>(specialist)"]
        policy_agent["policy_agent<br/>(specialist)"]
        supervisor -- delegate --> issue_analyst
        supervisor -- delegate --> policy_agent
        issue_analyst -- similar issues + outcomes --> supervisor
        policy_agent -- triage policy + on-caller prefs --> supervisor
    end

    subgraph oracle ["Oracle AI Database"]
        direction TB
        vs[(OracleVS<br/>issue reports + policy)]
        store[(AsyncOracleStore<br/>engineer prefs)]
        saver[(AsyncOracleSaver<br/>thread checkpoints)]
        cache[(OracleSemanticCache<br/>LLM call cache)]
    end

    issue_analyst -- search_similar_issues --> vs
    policy_agent  -- get_triage_policy    --> vs
    policy_agent  -- get_user_memory      --> store
    supervisor    -- checkpoint           --> saver
    cache_demo    -- application cache    --> cache

    supervisor --> answer["Triage recommendation<br/>severity · label · first-responder · references"]
```

## 14. The supervisor pattern and how it uses memory

### What the supervisor does

`langgraph_supervisor.create_supervisor(...)` builds an orchestrator that
is itself a LangGraph agent. The specialists are exposed to the supervisor
as **handoff tools**: when the supervisor decides *"I need issue data,"*
it emits a tool call that hands control to the `issue_analyst` sub-graph.
The specialist runs its own ReAct loop, returns a single summary message,
and control returns to the supervisor — which can now decide to call the
other specialist, or to synthesise the final answer.

This pattern means the supervisor is doing **planning and synthesis**, not
data fetching, and the specialists are doing **focused retrieval**, not
end-to-end reasoning.

### How the supervisor uses memory

The compiled graph receives both a checkpointer *and* a store:

```python
supervisor = supervisor_graph.compile(
    checkpointer=saver,    # AsyncOracleSaver — per-thread short-term memory
    store=agent_store,     # AsyncOracleStore — cross-thread long-term memory
)
```

- **Per-thread (short-term) memory — `AsyncOracleSaver`.** Every step of
  the supervisor *and* each specialist gets checkpointed, keyed by
  `thread_id`. Re-invoking with the same `thread_id` resumes
  mid-conversation; invoking with a fresh `thread_id` starts clean.
- **Cross-thread (long-term) memory — `AsyncOracleStore`.** Persists facts
  the system should know regardless of which conversation is happening.
  We pre-seeded user-scoped preferences under
  `("users", user_id, "memories")`. The `policy_agent`'s `get_user_memory`
  tool reads them from a fresh `thread_id` just as easily as from a
  returning one — which is what makes "long-term".

### A note on tools

The specialists don't reach into Oracle directly — they call **tools**. A tool is just a Python function the LLM is allowed to invoke: the model sees the function's name, signature, and docstring, decides *when* to call it and *with what arguments*, and gets the return value back as an observation to reason over. That think → call-a-tool → read-the-result loop is the entire ReAct pattern.

LangChain's **`@tool`** decorator is what turns an ordinary function into one the model can call: it introspects the type hints and docstring to generate the JSON schema the LLM needs, and wraps the function so its output is fed back into the conversation. The docstring matters — it's the "API doc" the model reads to decide whether the tool fits the task, so we write each one as an instruction, not an afterthought. See [LangChain · Tools](https://docs.langchain.com/oss/python/langchain/tools) for the full decorator reference.

## 15. Define `search_similar_issues` for the `issue_analyst`

The `issue_analyst` only needs to look things up in `OracleVS`. One LangChain `@tool` wraps `oracle_vs.similarity_search` so the agent can call it with a natural-language query.


In [31]:
from langchain_core.tools import tool

TOOL_CALL_COUNTS = {"issue_search": 0, "policy": 0, "user_memory": 0}

@tool
def search_similar_issues(query: str) -> str:
    """Search the historical issue corpus by semantic similarity (natural-language query)."""
    TOOL_CALL_COUNTS["issue_search"] += 1
    docs = oracle_vs.similarity_search(
        query, k=5, filter={"type": "issue_report"}
    )
    if not docs:
        return "No matches."
    return "\n\n---\n\n".join(d.page_content for d in docs)


## 16. Define `get_triage_policy` and `get_user_memory` for the `policy_agent`

The `policy_agent` has two distinct lookups, so two tools:

- **`get_triage_policy`** — pulls the standing triage policy from `OracleVS` via a focused similarity search.
- **`get_user_memory`** — pulls the active on-call engineer's saved preferences from `AsyncOracleStore`, scoped by `user_id`.


In [32]:
@tool
def get_triage_policy() -> str:
    """Fetch the standing on-call triage policy from OracleVS."""
    TOOL_CALL_COUNTS["policy"] += 1
    docs = oracle_vs.similarity_search(
        "on-call triage policy", k=1, filter={"type": "policy"}
    )
    return docs[0].page_content if docs else "No policy on file."


@tool
async def get_user_memory(user_id: str) -> str:
    """Look up long-term saved preferences for an on-call engineer by their user_id (cross-thread)."""
    TOOL_CALL_COUNTS["user_memory"] += 1
    items = await agent_store.asearch(
        ("users", user_id, "memories"),
        query="preference",
        limit=3,
    )
    if not items:
        return f"No saved memories for user_id={user_id!r}."
    return "\n".join(item.value.get("note", "") for item in items)


> **Key takeaway — small, focused tool surfaces.** Each specialist gets
> exactly the tools its job needs. That keeps the system prompt tight and
> reduces "tool-hallucination" (the LLM inventing tools that don't
> exist).

## 17. Compile the two specialist agents

`langchain.agents.create_agent` builds a ReAct-style agent — an LLM that
loops between "think" and "call a tool" until it decides to answer. We
give each specialist a focused `system_prompt` and a `name` (the
supervisor uses the name as the tool name when delegating).

In [33]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic

# One Claude Opus 4.8 model (adaptive thinking) drives the supervisor and both
# specialists. `chat_model_kwargs()` sets max_tokens=8192 + thinking=adaptive.
agent_model = ChatAnthropic(model=LLM_MODEL, **chat_model_kwargs())
print(f"agent model ready: {LLM_PROVIDER}/{LLM_MODEL}")

agent model ready: anthropic/claude-opus-4-8


In [35]:
issue_analyst = create_agent(
    agent_model,
    tools=[search_similar_issues],
    system_prompt=(
        "You are an issue analyst on an AI / agent platform team. Given a new "
        "incoming issue (title + body), use `search_similar_issues` to find the "
        "most relevant past issues. Return a concise 3–5 sentence summary: "
        "the closest 1–3 prior issues by id, each one's state (open vs closed) "
        "and any resolution detail that is actually present in its body, and any "
        "pattern you can extract (recurring root cause, common label, etc.). "
        "Do NOT invent a resolution that isn't in the retrieved text — if an "
        "issue is closed but the fix isn't stated, say so."
    ),
    name="issue_analyst",
)
print("issue_analyst compiled")

issue_analyst compiled


Now the policy specialist. It gets **two** tools — one that pulls the
standing triage policy from `OracleVS`, and one that fetches the active
on-call engineer's saved preferences from `AsyncOracleStore` (long-term
cross-thread memory). Two different memory backends feed the same model,
Claude Opus 4.8 via `ChatAnthropic`.

In [36]:
policy_agent = create_agent(
    agent_model,
    tools=[get_triage_policy, get_user_memory],
    system_prompt=(
        "You are the policy and preference agent. Use `get_triage_policy` for the "
        "standing triage policy, and `get_user_memory(user_id=...)` for the active "
        "on-call engineer's saved preferences (the user_id is mentioned in the "
        "supervisor's request). Return both verbatim \u2014 do not editorialise."
    ),
    name="policy_agent",
)
print(f"specialists compiled on {LLM_PROVIDER}/{LLM_MODEL}")


specialists compiled on anthropic/claude-opus-4-8


## 18. Compile the supervisor

The supervisor sees both specialists as **handoff tools**. We give it a
prompt that tells it the standard decomposition — *"call `policy_agent`,
then `issue_analyst`, then synthesise"* — and compile the whole graph
with the checkpointer (short-term) and store (long-term) wired in.

In [37]:
from langgraph_supervisor import create_supervisor

supervisor_graph = create_supervisor(
    [issue_analyst, policy_agent],
    model=agent_model,
    prompt=(
        "You are the on-call triage supervisor for an AI / agent platform team. "
        "For any incoming issue:\n"
        "1. Delegate to `policy_agent` for the standing triage policy + the active "
        "on-call engineer's user_id memories.\n"
        "2. Delegate to `issue_analyst` for similar past issues + their state/outcome.\n"
        "3. Synthesise a triage recommendation with five fields: **Severity** (P0/P1/P2/P3), "
        "**Label** (one or more), **First-responder** (a user_id, respecting the policy + "
        "the on-caller's preferences), **Similar past issues** (1–3 ids with one-line "
        "summaries), and **Suggested first action** (one sentence). Cite the policy and "
        "each reference inline."
    ),
)

The final wiring step: compile the supervisor graph with **both** memory
backends attached. The checkpointer (`saver`) handles short-term
per-thread state; the store (`agent_store`) handles long-term
cross-thread facts. This single `.compile()` call is where the agent's
two memory horizons converge into one runnable.

In [38]:
supervisor = supervisor_graph.compile(
    checkpointer=saver,    # per-thread short-term memory
    store=agent_store,     # cross-thread long-term memory
)
print(f"multi-agent supervisor compiled on {LLM_PROVIDER}/{LLM_MODEL}")

multi-agent supervisor compiled on anthropic/claude-opus-4-8


In [ ]:
# View the ACTUAL compiled graph LangGraph built (supervisor + handoff nodes),
# rendered straight from the runnable via its built-in mermaid support.
from IPython.display import Image, display

try:
    display(Image(supervisor.get_graph().draw_mermaid_png()))
except Exception as exc:
    # draw_mermaid_png() calls the mermaid.ink renderer; fall back to the text
    # definition (always works, no network) if that's unavailable.
    print(f"(image render unavailable: {exc})\n")
    print(supervisor.get_graph().draw_mermaid())

### The agents and their tools

The auto-generated graph above shows LangGraph's wiring; this view shows the **division of labour** — which agent owns which tool, and which Oracle store each tool reads:

```mermaid
flowchart TD
    sup{{"supervisor<br/>plans + synthesises"}}
    sup -->|handoff| ia["issue_analyst"]
    sup -->|handoff| pa["policy_agent"]
    ia -->|"search_similar_issues"| vs[("OracleVS<br/>issue reports")]
    pa -->|"get_triage_policy"| vs
    pa -->|"get_user_memory"| store[("AsyncOracleStore<br/>engineer prefs")]
    ia -.summary.-> sup
    pa -.summary.-> sup
```

`issue_analyst` gets one tool (semantic search over past issues); `policy_agent` gets two (the standing policy, and the on-caller's saved preferences). Small, focused tool surfaces keep each specialist's prompt tight and its behaviour predictable.

> **Key takeaway — one `.compile()`, two memory layers.** 
>
> Passing both
> `checkpointer=` and `store=` to compile is how LangGraph apps acquire
> both kinds of memory. 
>
> Either alone is incomplete: without the
> checkpointer you can't resume a conversation; without the store you
> can't remember anything across conversations.

## 19. Run the supervisor end-to-end

The request mentions the on-call engineer's `user_id` (`alice`). 

Watch the supervisor delegate to `policy_agent` (which retrieves the triage policy and **Alice's** saved preference from `AsyncOracleStore`) and `issue_analyst` (which retrieves similar past issues from `OracleVS`), then synthesise a triage recommendation that respects both. We generate a fresh `thread_id` on every execution so a notebook re-run cannot inherit old messages; every step is still checkpointed to `AsyncOracleSaver`. 

Reuse the printed id deliberately when you want to resume that exact conversation.

In [39]:
NEW_REQUEST = (
    "I'm on-call this week, user_id=alice. A new issue just landed:\n\n"
    "Title: load_dataset() fails on a public parquet dataset with "
    "KeyError on columns metadata\n"
    "Body: Calling load_dataset('some/public-parquet-dataset') raises "
    "KeyError '_data_files' after the download step. Reproducible on "
    "datasets 2.18 and 2.19. Other parquet datasets load fine. Happens "
    "on Linux and macOS; Python 3.11.\n\n"
    "Triage it."
)



thread_id=triage-parquet-6182f446f45f4243a47b261f124ef279

## Final triage recommendation

- **Severity:** **P1** — a *public* parquet dataset is unloadable, reproduces across datasets 2.18/2.19 and both Linux + macOS, with no known user-side workaround. Per policy [triage policy: P1 = "a popular dataset is unloadable," assign to primary on-call, resolve/escalate within 4h].

- **Label:** **`bug`** — a dataset fails to load/parse [triage policy: label-to-team routing, `'bug'` → primary on-call].

- **First-responder:** **alice** (primary on-call this week) — she owns `bug`-labelled load/parse/stream failures by default [user_id=alice memory].

- **Similar past issues:**
  - **hf-datasets-2943** (closed) — cache backwards-compat break; workaround = clear `~/.cache/huggingface/datasets/...`. Closest documented prior with a real fix.
  - **hf-datasets-2924** (open) — `load_dataset()` fails during download step (file-lock name error).
  - **hf-datasets-2937** (open) — `load_dataset()` down

/Users/richmondalake/Desktop/projects/OAMP/.venv313/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3747: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


Each `ainvoke` is scoped to a **`thread_id`**, and `AsyncOracleSaver` checkpoints the whole conversation under it. Reusing a `thread_id` *resumes* that conversation — the agent replays its prior messages and continues — so we mint a **fresh** `uuid` per run below to guarantee a clean start that can't inherit stale state from an earlier execution. Copy the printed id and pass it back in deliberately when you *want* to continue that exact thread (for example, a follow-up question after the first recommendation).

In [ ]:
THREAD_ID = f"triage-parquet-{uuid.uuid4().hex}"

In [ ]:
result = await supervisor.ainvoke(
    {"messages": [{"role": "user", "content": NEW_REQUEST}]},
    config={"configurable": {"thread_id": THREAD_ID}},
)

In [ ]:
last = result["messages"][-1]
_t = getattr(last, "text", None)
final_text = _t() if callable(_t) else (_t if isinstance(_t, str) else str(last.content))

required_fields = ["Severity", "Label", "First-responder", "Similar past issues", "Suggested first action"]
present = [f for f in required_fields if f in final_text]
checkpoint = await saver.aget({"configurable": {"thread_id": THREAD_ID}})

print(f"thread_id={THREAD_ID}\n")
print(final_text)
print(f"\nrecommendation fields present: {present}")

Each tool bumps a counter in `TOOL_CALL_COUNTS` when the model invokes it, so the print below is a simple **observability check**: it confirms the multi-agent run actually hit Oracle the way we expect — `issue_search` (the analyst) plus `policy` and `user_memory` (the policy agent) — rather than the LLM answering from thin air. A zero anywhere would mean an agent skipped its tool.

In [ ]:
print(f"Oracle-backed tool calls: {TOOL_CALL_COUNTS}")
print(f"thread checkpointed to AsyncOracleSaver: {checkpoint is not None}")

> **Key takeaway — Part 5: every layer earns its keep.** The final recommendation
> simultaneously cites the policy threshold (from `OracleVS`), Alice's saved
> preference that she owns `bug`-labelled load failures (from
> `AsyncOracleStore`), and the specific historical issues that match the
> symptom (also from `OracleVS`). No single tool call could have produced
> that — the multi-agent decomposition is doing real work, and every Oracle
> primitive contributed a piece of the answer.

# Part 6 — Durable chat sessions with `OracleChatMessageHistory`

Not every LangChain app is a LangGraph state machine. 

Sometimes you just want a **plain chat history**: a `session_id`, a list of messages, full
persistence, no graph. 

That's `OracleChatMessageHistory`.

## 20. Use `OracleChatMessageHistory` as a standalone primitive

We pass the same `oracle_vs.client` so all three `langchain-oracledb`
primitives (vector store, cache, chat history) share **one** Oracle
connection. We append a few turns, then "simulate a process restart" by
opening a fresh handle on the same `session_id` and reading them back.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_oracledb.chat_message_histories import OracleChatMessageHistory

session_id = "oncall-bob-2026-W22"
history = OracleChatMessageHistory(
    session_id=session_id,
    client=oracle_vs.client,          # same sync connection as OracleVS + the cache
    table_name="langchain_oncall_chat",
)

history.clear()  # idempotent: every run of this cell starts the session clean

history.add_messages([
    HumanMessage(content="What was last week's triage call on the LangGraph loop issue?"),
    AIMessage(content="Routed AGT-101 to Bob with severity P1; resolved via iteration cap + state dedup."),
])

# Construct a genuinely new Python handle, as another request/process would.
reloaded_history = OracleChatMessageHistory(
    session_id=session_id,
    client=oracle_vs.client,
    table_name="langchain_oncall_chat",
)
persisted_messages = reloaded_history.messages

print(f"reloaded {len(persisted_messages)} messages under session_id={session_id!r}")
for msg in persisted_messages:
    print(f"  {type(msg).__name__:13s} {msg.content[:120]}")
print("\na fresh OracleChatMessageHistory handle recovered the transcript:",
      reloaded_history is not history and len(persisted_messages) == 2)

reloaded 2 messages under session_id='oncall-bob-2026-W22'
  HumanMessage  What was last week's triage call on the LangGraph loop issue?
  AIMessage     Routed AGT-101 to Bob with severity P1; resolved via iteration cap + state dedup.

a fresh OracleChatMessageHistory handle recovered the transcript: True


> **Key takeaway — Part 6: pick the right primitive.** Use the LangGraph
> checkpointer + store when you have a multi-step agent; use
> `OracleChatMessageHistory` when you have a plain chatbot with no graph.
> Both persist to the same Oracle database and can share a connection — so
> you can start with a simple chat history and graduate to the full agent
> stack without changing your data backend.

# Part 7 — Teardown

We opened three Oracle connections and created several tables. The final cell
deletes this run's checkpoint thread, drops the demo tables (`OracleVS`, the
semantic cache, chat history, and the long-term store), and closes all three
connections — so a re-run starts clean. Set `KEEP_ORACLE_TABLES=1` to keep the
tables for inspection while still removing the checkpoint thread.

## 21. Close connections

In [41]:
KEEP_ORACLE_TABLES = os.environ.get("KEEP_ORACLE_TABLES", "0").lower() in {"1", "true", "yes"}

# 1) This run's per-thread checkpoints.
await saver.adelete_thread(THREAD_ID)
print("deleted checkpoint thread:", THREAD_ID)

# 2) The demo tables (unless you want to inspect them).
if KEEP_ORACLE_TABLES:
    print("KEEP_ORACLE_TABLES=1 — leaving OracleVS / cache / chat / store tables in place.")
else:
    await agent_store.ateardown()                       # long-term store tables
    with oracle_client.cursor() as cur:
        for tbl in (VECTOR_TABLE, "oncall_semantic_cache", "langchain_oncall_chat"):
            try:
                cur.execute(f'DROP TABLE "{tbl}" PURGE')
            except oracledb.DatabaseError as exc:
                if getattr(exc.args[0], "code", None) != 942:  # ORA-00942 = already gone
                    raise
    oracle_client.commit()
    print("dropped OracleVS, semantic-cache, chat-history, and long-term store tables")

# 3) Close all three connections.
oracle_client.close()
await saver_conn.close()
await store_conn.close()
print("✅ connections closed; teardown complete")

deleted checkpoint thread: triage-parquet-6182f446f45f4243a47b261f124ef279
dropped OracleVS, semantic-cache, chat-history, and long-term store tables
✅ connections closed; teardown complete


> **Key takeaway — Part 7: single substrate, full lifecycle.** One database,
> five LangChain/LangGraph primitives, two memory horizons (per-thread +
> cross-thread), a semantic cache, and a separate chat-history store — all
> provisioned, exercised, and torn down from one notebook. That's the payoff of
> using Oracle AI Database as the agent's whole substrate: the vector knowledge
> base, both memory layers, the LLM cache, and the chat log are one system to
> provision, secure, back up, and reason about — not five.